In [1]:
!pip install -q -U keras-tuner


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 9.7 MB/s eta 0:00:00


In [2]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf
from sklearn.model_selection import KFold
from matplotlib import pyplot as plt
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score, roc_curve
import numpy as np
import keras_tuner as kt  # Keras Tuner

ValueError: mount failed

In [ ]:
# Set global seeds for reproducibility
seed = 123
tf.random.set_seed(seed)
np.random.seed(seed)

In [ ]:
data_dir = '/content/drive/My Drive/breakhis'  # Replace with your actual path to the dataset


In [ ]:
from keras.saving import register_keras_serializable

@register_keras_serializable()
class PAM(tf.keras.layers.Layer):
    """Position Attention Module"""
    def __init__(self):
        super(PAM, self).__init__()

    def build(self, input_shape):
        self.query_conv = tf.keras.layers.Conv2D(input_shape[-1] // 8, kernel_size=1)
        self.key_conv = tf.keras.layers.Conv2D(input_shape[-1] // 8, kernel_size=1)
        self.value_conv = tf.keras.layers.Conv2D(input_shape[-1], kernel_size=1)
        self.gamma = self.add_weight(name="gamma", shape=(), initializer="zeros", trainable=True)

    def call(self, inputs):
        query = self.query_conv(inputs)  # [batch, h, w, c//8]
        key = self.key_conv(inputs)  # [batch, h, w, c//8]
        value = self.value_conv(inputs)  # [batch, h, w, c]

        # Compute attention
        query = tf.reshape(query, [tf.shape(query)[0], -1, tf.shape(query)[-1]])  # [batch, hw, c//8]
        key = tf.transpose(tf.reshape(key, [tf.shape(key)[0], -1, tf.shape(key)[-1]]), perm=[0, 2, 1])  # [batch, c//8, hw]
        energy = tf.matmul(query, key)  # [batch, hw, hw]
        attention = tf.nn.softmax(energy, axis=-1)  # Spatial attention

        value = tf.reshape(value, [tf.shape(value)[0], -1, tf.shape(value)[-1]])  # [batch, hw, c]
        out = tf.matmul(attention, value)  # [batch, hw, c]
        out = tf.reshape(out, tf.shape(inputs))  # Restore shape
        return self.gamma * out + inputs


@register_keras_serializable()
class CAM(tf.keras.layers.Layer):
    """Channel Attention Module"""
    def __init__(self):
        super(CAM, self).__init__()

    def build(self, input_shape):
        self.gamma = self.add_weight(name="gamma", shape=(), initializer="zeros", trainable=True)

    def call(self, inputs):
        # Compute attention
        query = tf.reshape(inputs, [tf.shape(inputs)[0], -1, tf.shape(inputs)[-1]])  # [batch, hw, c]
        key = tf.transpose(query, perm=[0, 2, 1])  # [batch, c, hw]
        energy = tf.matmul(key, query)  # [batch, c, c]
        attention = tf.nn.softmax(energy, axis=-1)  # Channel attention

        value = tf.reshape(inputs, [tf.shape(inputs)[0], -1, tf.shape(inputs)[-1]])  # [batch, hw, c]
        out = tf.matmul(value, attention)  # [batch, hw, c]
        out = tf.reshape(out, tf.shape(inputs))  # Restore shape
        return self.gamma * out + inputs




In [ ]:
@register_keras_serializable()
class DANetBlock(tf.keras.layers.Layer):
    """Dual Attention Network Block"""
    def __init__(self):
        super(DANetBlock, self).__init__()
        self.pam = PAM()
        self.cam = CAM()

    def call(self, inputs):
        pam_out = self.pam(inputs)
        cam_out = self.cam(inputs)
        return pam_out + cam_out  # Combine PAM and CAM outputs


In [ ]:
def build_model():
    inputs = tf.keras.Input(shape=(256, 256, 3))

    # Convolutional Block 1
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # Convolutional Block 2
    x = tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # Convolutional Block 3
    x = tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # DANet Block
    x = DANetBlock()(x)

    # Global Average Pooling and Fully Connected Layers
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    # Output Layer
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    # Model Compilation
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    learning_rate = 0.0004
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [ ]:
# Test DANetBlock
x_test = tf.random.normal((1, 64, 64, 256))  # Example input
danet_block = DANetBlock()
x_out = danet_block(x_test)

print(f"Output shape: {x_out.shape}")


In [ ]:

# Step 3: Load and prepare datasets with a given batch size
img_size = (256, 256)


def prepare_datasets(batch_size):
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    val_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    # Data Augmentation and Normalization
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.2),
        tf.keras.layers.RandomContrast(0.2),
        tf.keras.layers.Lambda(lambda x: tf.image.random_brightness(x, max_delta=0.3)),
        tf.keras.layers.Lambda(lambda x: tf.image.random_saturation(x, lower=0.8, upper=1.2)),
    ])

    normalization_layer = tf.keras.layers.Rescaling(1./255)
    train_dataset = train_dataset.map(lambda x, y: (data_augmentation(normalization_layer(x), training=True), y))
    val_dataset = val_dataset.map(lambda x, y: (normalization_layer(x), y))

    return train_dataset, val_dataset

In [ ]:

# Train and evaluate the model
batch_size = 16
train_dataset, val_dataset = prepare_datasets(batch_size=batch_size)

In [ ]:
model = build_model()
history = model.fit(train_dataset, validation_data=val_dataset, epochs=30)



In [ ]:
# Plot Loss and Accuracy During Training
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Evaluate the model on validation dataset
y_true = []
y_pred_probs = []

for batch in val_dataset.as_numpy_iterator():
    X, y = batch
    preds = model.predict(X)
    y_true.extend(y)
    y_pred_probs.extend(preds)

In [ ]:
# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs).flatten()
y_pred = (y_pred_probs > 0.5).astype(int)

In [ ]:
# Compute metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_pred_probs)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC-ROC: {roc_auc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Class 0", "Class 1"])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

# Plot PR Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred_probs)
pr_auc = np.trapz(precision_vals, recall_vals)
plt.figure(figsize=(10, 6))
plt.plot(recall_vals, precision_vals, label=f'PR Curve (AUC = {pr_auc:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="upper right")
plt.grid()
plt.show()

In [ ]:
import os

In [ ]:
# Save the trained model
model_path = '/content/drive/My Drive/saved_model/cnn_model_breast_danet_data.keras'
model.save(model_path)
print(f"Model saved at: {model_path}")

In [ ]:
# Save predictions for DANet
model_path1 = '/content/drive/My Drive/saved_models/danet_y_true.npy'

np.save(model_path1, y_true)
